# Загружаем данные

Загружаем граф из файла, получая список рёбер, множество вершин и список смежности.

In [1]:
def load_graph(file, directed=False):
    edges = []
    nodes = set()
    adjacency = {}

    with open(file, 'r') as file:
        for line in file:
            # skip comments and empty lines
            if line.startswith('#') or not line.strip():
                continue
            u, v = map(int, line.strip().split())

            edges.append((u, v))
            nodes.update([u, v]) # add both nodes to the set

            if u not in adjacency:
                adjacency[u] = set()
            adjacency[u].add(v)

            if not directed:
                if v not in adjacency:
                    adjacency[v] = set()
                adjacency[v].add(u)

    return edges, nodes, adjacency

# Вычисление расстояний между вершинами сети

## landmarks_basic

Для начала сделаем функцию, которая будет из всех вершин будет выбирать ориентиры. В самом простом случае берём рандомные k вершин:

In [2]:
import random

def select_landmarks(nodes, k):
    len_nodes = len(nodes)
    if k > len_nodes:
        raise ValueError(f"you provided k={k} landmarks, but the graph has only {len_nodes} nodes")
    return random.sample(list(nodes), k)

Затем сделаем функцию для BFS(source vertex), которая по списку смежности будет определять словарь distances. В нём distance[v] -> расстояние от source до v.

In [3]:
from collections import deque

def run_bfs_n_return_distances(adjacency, source):
    distances = {source: 0}
    queue = deque([source])

    while queue: # is not empty
        u = queue.popleft()
        adjacency_u = adjacency.get(u, [])
        for v in adjacency_u:
            if v not in distances:
                distances[v] = distances[u] + 1
                queue.append(v)
    return distances

Далее сделаем функцию, которая будет запускать BFS для всех ориентиров и заполнять словарь landmark_distances. В нём landmark_distances[l] -> distances (словарь из пред. пункта) от l до всех остальных вершин:

In [4]:
def compute_landmark_distances(adjacency, landmarks):
    landmark_distances = {}
    for cur_landmark in landmarks:
        landmark_distances[cur_landmark] = run_bfs_n_return_distances(adjacency, source=cur_landmark)
    return landmark_distances

После этого сделаем функцию, которая будет оценивать расстояние между вершинами s и t по формуле:
$$
d(s, t) \approx \min\limits_{l \in L} (ld[l][s] + ld[l][t])
$$

In [5]:
def estimate_distance(s, t, landmark_distances):
    all_estimates = []
    for l, ld_l in landmark_distances.items():
        ds = ld_l.get(s, float('inf'))
        dt = ld_l.get(t, float('inf'))
        all_estimates.append(ds + dt)
    ans = min(all_estimates) if all_estimates else float('inf')
    return ans

Наконец, можем сделать финальную функцию landmarks_basic, которая использует все предыдущие функции-помощники: среди всех вершин выделяет 'k' ориентиров, вычисляет расстояния от каждого ориентира до всех остальных вершин графа, а затем считает расстояние между двумя вершинами по формуле из предыдущего пункта.

In [6]:
def landmarks_basic(adjacency, nodes, s, t, k):
    landmarks = select_landmarks(nodes, k)
    landmark_distances = compute_landmark_distances(adjacency, landmarks)
    distance = estimate_distance(s, t, landmark_distances)
    return distance

## Исследование работы алгоритмов

Давайте проверим работу этого алгоритма на разных графах.

Определим мета-параметры:

In [7]:
graph_path = "datasets/undirected/CA-AstroPh.txt"
k = 10
distance_estimator_algorithm = landmarks_basic
num_sampled_pairs = 100

Загрузим граф:

In [8]:
graph = load_graph(graph_path, directed=False)
[edges, nodes, adjacency] = graph
print(f"загружен граф с {len(nodes)} вершинами и {len(edges)} рёбрами")

загружен граф с 18772 вершинами и 396160 рёбрами


Выберем пары случайных вершин:

In [9]:
pairs = []
nodes_list = list(nodes)
for _ in range(num_sampled_pairs):
    s, t = random.sample(nodes_list, 2)
    pairs.append((s, t))

И посчитам: (реальное расстояние, оценка расстояния):

In [10]:
distances_info = []

for s, t in pairs:
    exact_distance = run_bfs_n_return_distances(adjacency, s).get(t, float('inf'))
    estimated_distance = distance_estimator_algorithm(adjacency, nodes, s, t, k)
    distances_info.append((exact_distance, estimated_distance))

После этого посчитаем разные метрики для оценки точности работы алгоритма:

In [11]:
import math

# filter out invalid (inf or nan) values
valid_distances_info = [
    (e, f) for e, f in distances_info
    if math.isfinite(e) and math.isfinite(f)
]

abs_errors = [abs(e - f) for e, f in valid_distances_info]

Посчитаем MAE (Mean Absolute Error). Данная метрика не очень чувствительна к выбросам. Чем она меньше, тем, очевидно, лучше. MAE = 0 - идеальный случай. Также посчитаем минимальную и максимальную ошибки:

In [12]:
min_err = min(abs_errors)
max_err = max(abs_errors)
mae = sum(abs_errors) / len(abs_errors)
print("min error:", min_err)
print("max error:", max_err)
print("MAE:", mae)

min error: 0
max error: 4
MAE: 2.3372093023255816
